# Portfolio 03｜LangGraphを利用したAIエージェント

LangGraphの`State`・`Node`・`Edge`・`Conditional Edge`を利用し、
ユーザーの質問に応じてRAGまたはWeb検索を選択して回答を生成するAIエージェントを構築します。

本Portfolioでは、これまで作成したRAGとWeb検索をLangGraphのNodeとして統合し、
回答生成後にReflectionを行うワークフローを実装します。

GitHub: https://github.com/masatoppp

## 全体構成

```text
START
  ↓
route
  ↓
Conditional Edge
 ↙        ↘
RAG       Web
 ↘        ↙
 answer
   ↓
reflection
   ↓
Conditional Edge
 ↙        ↘
retry      END
 ↓
route
```


## 1. ライブラリの読み込み


In [ ]:
from pathlib import Path
from datetime import datetime
import time

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from tavily import TavilyClient


## 2. Stateの定義

`AgentState`に、Graph全体で受け渡す情報を定義します。

- `query`：ユーザーの質問
- `route`：RAG / Webの判定結果
- `context`：RAGまたはWeb検索から取得した情報
- `answer`：LLMが生成した回答
- `reflection`：回答評価結果
- `iteration`：再生成回数
- `has_answer`：取得した情報から回答できたかどうか
- `sources`：RAGで参照した文書の出典


### ダミーデータの配置

RAGで使用するダミー社内文書は、Portfolio 01で作成したデータを再利用しています。
GitHub上で単体実行できるよう、本リポジトリにも`RAG_test_data`フォルダを配置し、
Notebookからは相対パスで読み込みます。

```python
base_path = Path("RAG_test_data")
```


In [ ]:
class AgentState(BaseModel):
    query: str = Field(..., description="ユーザーからの質問")
    route: str = Field(default="", description="選択された処理ルート")
    context: str = Field(default="", description="RAGやWeb検索で取得した情報")
    answer: str = Field(default="", description="生成された回答")
    reflection: str = Field(default="", description="回答に対する評価結果")
    iteration: int = Field(default=0, description="再生成した回数")
    has_answer: bool = Field(
        default=True,
        description="参考情報から質問に回答できるかどうか"
    )
    sources: list[str] = Field(
        default_factory=list,
        description="RAGで参照した文書の出典"
    )


## 3. ローカルLLMの読み込み

Qwen3-4Bを4bit量子化してローカルGPU上で実行します。

当初はQwen3-1.7Bも検証しましたが、Reflection判定の安定性を高めるため、
本PortfolioではQwen3-4Bを使用しています。

Route判定やReflectionでは短い出力、回答生成では長い出力が必要になるため、
`generate_qwen()`で`max_new_tokens`を引数として指定できるようにします。


In [ ]:
model_name = "Qwen/Qwen3-4B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)


In [4]:
def generate_qwen(
    prompt: str,
    max_new_tokens: int = 128
) -> str:

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens
    )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return response

## 4. Route Node

ユーザーの質問を確認し、社内文書を参照する質問なら`rag`、
外部の情報や最新情報が必要な質問なら`web`を選択します。

Route Nodeは判定結果だけをStateへ返します。

### Router設計について

本構成では、最初に質問の種類からRAGまたはWeb検索を選択するRouter型としています。

社内制度・勤怠・福利厚生などの質問はRAGへ振り分け、
RAGに回答根拠が存在しない場合でも自動的にWeb検索へ切り替えません。

これは、社内文書に根拠がない内容をWeb上の一般的な情報で補完し、
会社固有の制度として誤って回答することを防ぐためです。

一方、最新ニュースや社外サービス、技術動向などはWeb検索へ振り分けます。


In [5]:
def route_node(state: AgentState):
    prompt = f"""
    ユーザーの質問を確認し、次の基準で 'rag' または 'web' を選択してください。

    'rag':
    - 社内制度、社内規定、社内手続き
    - 有給休暇、勤怠、給与、福利厚生
    - 社用PCの紛失、故障、盗難、利用トラブル
    - 情報システム部門や社内運用に関する質問

    'web':
    - 最新ニュース
    - 現在発生している不具合
    - 社外製品・サービス・技術動向
    - インターネット上の最新情報が必要な質問

    必ず 'rag' または 'web' のどちらかだけを回答してください。

    ユーザーの質問:
    {state.query}
    """

    result = generate_qwen(
        prompt,
        max_new_tokens=32
    ).lower()

    if "rag" in result:
        route = "rag"
    elif "web" in result:
        route = "web"
    else:
        route = "web"

    return {
        "route": route
    }

def select_route(state: AgentState):
    return state.route

## 5. RAG構築について

本Portfolioで使用するRAG用データおよび基本的なRAG構成は、
既存Portfolio **「metadata-rag-demo」** で実装したものを再利用しています。

使用する社内文書データ、チャンク分割、Embedding、Chroma、Retrieverの基本構成は前作と同一です。

本PortfolioではRAG自体の再実装を目的とせず、
既存のRAG処理をLangGraphのNodeとして組み込み、
Web検索・回答生成・Reflectionと組み合わせてワークフローを制御することを主眼としています。


In [6]:
folder_path = Path("RAG_test_data")
txt_files = list(folder_path.rglob("*.txt"))

documents = []

for file_path in txt_files:
    text = file_path.read_text(encoding="utf-8")
    relative_path = file_path.relative_to(folder_path)

    metadata = {
        "path": str(relative_path),
        "filename": file_path.name
    }

    document = Document(
        page_content=text,
        metadata=metadata
    )

    documents.append(document)

print(f"文書数: {len(documents)}")


文書数: 22


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"チャンク数: {len(chunks)}")


チャンク数: 31


In [ ]:
embedding_function = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_function
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


In [9]:
def rag_node(state: AgentState):

    results = retriever.invoke(state.query)

    context = "\n\n".join(
        f"""【資料】
ファイル名: {doc.metadata['filename']}
保存場所: {doc.metadata['path']}

【本文】
{doc.page_content}
"""
        for doc in results
    )

    sources = []
    for doc in results:
        source = (
            f"ファイル名: {doc.metadata['filename']}\n"
            f"保存場所: {doc.metadata['path']}"
        )
        if source not in sources:
            sources.append(source)

    return {
        "context": context,
        "sources": sources
    }

## 6. Web検索について

Web検索処理は、既存Portfolio **「local-llm-websearch-chat」** で実装した
Tavily検索を再利用します。

本Portfolioでは、検索処理を`web_node`としてLangGraphへ組み込みます。
また、「最新」などの質問で古い検索結果が優先されにくくなるよう、
検索クエリへ現在日付を追加します。

※ Tavily APIキーは事前に環境変数等で設定済みであることを前提とします。


In [10]:
tavily = TavilyClient()


In [11]:
def search_web(search_query: str) -> str:

    search_result = tavily.search(
        query=search_query,
        max_results=3
    )

    search_text = ""

    for result in search_result["results"]:
        search_text += (
            f"タイトル: {result['title']}\n"
            f"URL: {result['url']}\n"
            f"内容: {result['content']}\n\n"
        )

    return search_text


In [ ]:
def web_node(state: AgentState):

    current_date = datetime.now().strftime("%Y-%m-%d")
    search_query = f"{state.query} 最新 {current_date}"

    context = search_web(search_query)

    return {
        "context": context,
        "sources": []
    }


## 7. Answer Node

RAGまたはWeb検索で取得した`context`とユーザーの質問をLLMへ渡し、
最終回答を生成します。

回答が途中で切れることを避けるため、回答生成のみ`max_new_tokens=1024`としています。
また、検索結果に含まれない情報を必要以上に補わないよう、プロンプトで制約します。


In [13]:
def answer_node(state: AgentState):
    prompt = f"""
    以下の参考情報のみを使用して、ユーザーの質問に回答してください。

    ユーザーの質問:
    {state.query}

    参考情報:
    {state.context}

    回答ルール:
    - 参考情報に記載されている内容だけを使用してください。
    - 参考情報から推測して情報を補完しないでください。

    - Web検索結果を使用する場合は、検索結果に明記されている事実だけを回答してください。
    - 「おすすめ」「最高」「コスパが良い」などの主観的な評価を追加しないでください。
    - 最新ニュースを求められた場合は、一般的な業界動向ではなく、検索結果に含まれる具体的な出来事を優先してください。

    - 質問に直接必要な情報だけを回答してください。
    - 関連はあるが、質問への回答に不要な情報は含めないでください。
    - 質問に直接答える情報が存在しない場合は、
      「参照した情報には、質問に該当する情報がありません。」
      とだけ回答してください。
    - 回答は簡潔にまとめてください。
    """

    answer = generate_qwen(
        prompt,
        max_new_tokens=1024
    )

    no_answer_phrases = [
        "参考情報はありません",
        "情報がありません",
        "記載されていません",
        "該当する情報がありません",
        "質問に直接答える情報がありません"
    ]

    has_answer = not any(
        phrase in answer
        for phrase in no_answer_phrases
    )

    return {
        "answer": answer,
        "has_answer": has_answer
    }

## 8. Reflection Node

生成された回答がユーザーの質問に適切に答えているかを評価します。

- 適切な回答：`ok`
- 再生成が必要：`retry`

`retry`の場合は`iteration`を1増やし、Route Nodeへ戻します。
無限ループを避けるため、再生成は最大3回までとします。

また、Answer Nodeで「参照情報から回答できない」と判定した場合は、
正しい回答不能ケースとしてReflectionをスキップし、`ok`として終了します。


In [ ]:
def reflection_node(state: AgentState):

    if not state.has_answer:
        return {
            "reflection": "ok",
            "iteration": state.iteration
        }

    prompt = f"""
    回答を評価してください。

    【質問】
    {state.query}

    【参考情報】
    {state.context}

    【回答】
    {state.answer}

    次のどれか1つでも当てはまる場合は 'retry' と回答してください。

    - 回答に参考情報にない内容が含まれている
    - 回答に質問と関係のない内容が含まれている
    - 回答の中で内容が矛盾している

    問題がない場合だけ 'ok' と回答してください。

    'ok' または 'retry' のどちらかだけを回答してください。
    """

    result = generate_qwen(
        prompt,
        max_new_tokens=32
    ).lower()

    if "retry" in result:
        reflection = "retry"
        iteration = state.iteration + 1
    else:
        reflection = "ok"
        iteration = state.iteration

    return {
        "reflection": reflection,
        "iteration": iteration
    }


def select_reflection(state: AgentState):
    if state.reflection == "ok":
        return "ok"

    if state.iteration >= 3:
        return "end"

    return "retry"


## 9. Graphの構築

これまで作成したNodeを登録し、EdgeとConditional Edgeで処理順序を定義します。

```text
START
  ↓
route
  ↓
select_route
 ↙        ↘
rag        web
 ↘        ↙
   answer
     ↓
 reflection
     ↓
select_reflection
 ↙      ↓      ↘
retry   ok      end
 ↓      ↓        ↓
route   END      END
```

Jupyter NotebookではGraph構造を変更した場合、
`StateGraph(AgentState)`から作り直して再compileします。


In [15]:
workflow = StateGraph(AgentState)

workflow.add_node("route", route_node)
workflow.add_node("rag", rag_node)
workflow.add_node("web", web_node)
workflow.add_node("answer", answer_node)
workflow.add_node("reflection", reflection_node)

workflow.add_edge(START, "route")

workflow.add_conditional_edges(
    "route",
    select_route,
    {
        "rag": "rag",
        "web": "web"
    }
)

workflow.add_edge("rag", "answer")
workflow.add_edge("web", "answer")

workflow.add_edge("answer", "reflection")

workflow.add_conditional_edges(
    "reflection",
    select_reflection,
    {
        "retry": "route",
        "ok": END,
        "end": END
    }
)

app = workflow.compile()


## 10. 最終動作検証

任意の質問を入力し、Route判定からRAG / Web検索、回答生成、
ReflectionまでGraph全体が正常に実行されることを確認します。

あわせて、一連の処理に要した時間を計測します。

このセルはエージェント本体の実装ではなく、**完成したGraphの動作検証用**です。


In [20]:
query = input("質問を入力してください: ")

initial_state = AgentState(
    query=query
)

start_time = time.perf_counter()

result = app.invoke(initial_state)

elapsed_time = time.perf_counter() - start_time

print(f"\n【質問】\n{query}")
print(f"\n【Route】\n{result['route']}")
print("\n【回答】\n")
print(result["answer"])
print(f"\n【処理時間】\n{elapsed_time:.2f} 秒")

if result["route"] == "rag" and result["has_answer"]:
    print("\n【出典】")
    for source in result["sources"]:
        print(source)
        print()
            
print(f"\n【Reflection】\n{result['reflection']}")
print(f"\n【Iteration】\n{result['iteration']}")


質問を入力してください:  育児休業中の給与は何％支給されますか？



【質問】
育児休業中の給与は何％支給されますか？

【Route】
rag

【回答】

参照した情報には、質問に該当する情報がありません。

【処理時間】
2.21 秒

【Reflection】
ok

【Iteration】
0


## 11. まとめ

本Portfolioでは、LangGraphを利用して以下の処理を1つのワークフローとして構築しました。

1. `AgentState`による状態管理
2. Route NodeによるRAG / Web検索の選択
3. Conditional Edgeによる処理分岐
4. 既存RAGをNodeとして統合
5. Tavily Web検索をNodeとして統合
6. 取得したContextを利用した回答生成
7. `has_answer`による回答可能性の管理
8. Reflectionによる回答評価
9. `retry`時の再実行と`iteration`による上限管理
10. RAG回答時の参照元表示

RAGとWeb検索そのものは既存Portfolioの実装を再利用し、
本PortfolioではそれらをLangGraph上で接続し、
StateとConditional Edgeによってワークフローを制御することを中心に実装しました。

### 設計上の判断

社内情報としてRAGへ振り分けた質問については、
参照文書に回答根拠がない場合でもWeb検索へ自動Fallbackしません。

これは、Web上の一般情報を社内固有の制度として回答することを防ぐためです。
参照文書に根拠がない場合は、
「参照した情報には、質問に該当する情報がありません。」
と回答して終了します。

### 今後の改善点

- `has_answer`は現在、生成された回答文から判定しているため、回答可能性そのものを誤判定する可能性があります。
- Reflectionの評価理由を次回の回答生成へ直接渡していないため、retry時に同種の回答が再生成される場合があります。
- Web検索結果の鮮度や本文抽出品質によって回答精度が左右されます。
- 今後は、社内固有情報と一般情報をさらに分類し、一般情報として回答可能な場合のみRAGからWeb検索へFallbackする構成も検討できます。
